In [13]:
# Cell 0 — Project Imports

import nibabel as nib
import numpy as np
import torch

In [14]:
# Cell 1 — Synthetic NIfTI Array와 Affine 생성

def create_synthetic_nifti(
    array_shape_ijk: tuple[int, int, int],              # [I, J, K]
    voxel_spacing_ijk_mm: tuple[float, float, float],   # [I, J, K] mm
    origin_xyz_mm: tuple[float, float, float],          # [X, Y, Z] mm
) -> nib.Nifti1Image:
    """3D voxel array와 axis-aligned affine을 결합한 synthetic NIfTI 생성."""

    # 전체 voxel 개수 계산: 6 × 8 × 10 = 480 voxels
    number_of_voxels: int = int(
        np.prod(array_shape_ijk)
    )
    
    # 0부터 479까지의 값을 가진 1D array 생성 후, 
    # NIfTI 내부의 3D voxel array [I, J, K]로 변환
    voxel_array: np.ndarray = np.arange(
        number_of_voxels,
        dtype=np.float32,
    ).reshape(
        array_shape_ijk
    )  # [I=6, J=8, K=10]
    
    # Voxel index를 physical coordinate로 변환할 affine 생성
    affine_ijk_to_xyz: np.ndarray = np.eye(
        4,
        dtype=np.float64,
    ) # [4, 4]
    
    # Affine의 3×3 영역에 axis별 voxel spacing 기록 + physical origin
    affine_ijk_to_xyz[0, 0] = voxel_spacing_ijk_mm[0]
    affine_ijk_to_xyz[1, 1] = voxel_spacing_ijk_mm[1]
    affine_ijk_to_xyz[2, 2] = voxel_spacing_ijk_mm[2]
    affine_ijk_to_xyz[:3, 3] = np.asarray(
        origin_xyz_mm,
        dtype=np.float64
    )
    
    # Voxel intensity와 공간 geometry를 하나의 NIfTI 객체로 결합
    nifti_image = nib.Nifti1Image(
        voxel_array,
        affine_ijk_to_xyz,
    )
    
    return nifti_image
    
    
# Shape, spacing, origin이 알려진 synthetic medical image 생성
synthetic_nifti = create_synthetic_nifti(
    array_shape_ijk=(
        6,
        8,
        10,
    ),
    voxel_spacing_ijk_mm=(
        2.0,
        1.5,
        3.0,
    ),
    origin_xyz_mm=(
        -120.0,
        -90.0,
        -60.0,
    ),
)

# NIfTI에서 voxel intensity array 추출
nifti_voxel_array: np.ndarray = synthetic_nifti.get_fdata(
    dtype=np.float32,
)  # [I=6, J=8, K=10]

# Voxel index에서 physical coordinate로 가는 affine 추출
nifti_affine: np.ndarray = synthetic_nifti.affine  # [4, 4]

# Header에 기록된 axis별 voxel spacing 추출
header_spacing_ijk_mm: tuple[float, ...] = (
    synthetic_nifti.header.get_zooms()[:3]
)

# Affine에서 각 array axis의 physical 방향 해석
axis_codes: tuple[str | None, ...] = nib.aff2axcodes(
    nifti_affine
)

print("Array shape:", nifti_voxel_array.shape)
print("Array dtype:", nifti_voxel_array.dtype)
print("Header spacing (mm):", header_spacing_ijk_mm)
print("Axis codes:", axis_codes)
print("Affine:")
print(nifti_affine)

Array shape: (6, 8, 10)
Array dtype: float32
Header spacing (mm): (np.float32(2.0), np.float32(1.5), np.float32(3.0))
Axis codes: ('R', 'A', 'S')
Affine:
[[   2.     0.     0.  -120. ]
 [   0.     1.5    0.   -90. ]
 [   0.     0.     3.   -60. ]
 [   0.     0.     0.     1. ]]


In [15]:
# Cell 2 — Voxel Index를 Physical Coordinate로 변환

def voxel_index_to_physical_coordinate(
    voxel_index_ijk: torch.Tensor,       # [3] = [I, J, K]
    affine_ijk_to_xyz: torch.Tensor,     # [4, 4]
) -> torch.Tensor:                       # [3] = [X, Y, Z] mm
    """Voxel index를 affine으로 변환한 physical coordinate 반환."""
    
    # [I, J, K] → [I, J, K, 1]
    homogeneous_voxel_index: torch.Tensor = torch.cat(
        (
            voxel_index_ijk,
            torch.ones(
                1,
                dtype=voxel_index_ijk.dtype,
                device=voxel_index_ijk.device,
            ),
        )
    )  # [4]
    
    # Affine과 homogeneous voxel index의 matrix-vector multiplication
    # [4, 4] @ [4] → [4] 
    homogeneous_physical_coordinate: torch.Tensor = (
        affine_ijk_to_xyz
        @ homogeneous_voxel_index
    ) # [4]
    
    physical_coordinate_xyz_mm: torch.Tensor = (
        homogeneous_physical_coordinate[:3]
    ) # [3]
    
    return physical_coordinate_xyz_mm
    
    
# Cell 1에서 만든 NumPy affine을 계산용 PyTorch Tensor로 변환
affine_ijk_to_xyz = torch.from_numpy(
    synthetic_nifti.affine.copy()
).to(
    dtype=torch.float64
)  # [4, 4]

# Physical 위치를 확인할 voxel index 선택
voxel_index_ijk = torch.tensor(
    [
        2.0,
        3.0,
        4.0,
    ],
    dtype=torch.float64,
)  # [3] = [I, J, K]

# Voxel index [I, J, K]를 physical coordinate [X, Y, Z] mm로 변환
physical_coordinate_xyz_mm = (
    voxel_index_to_physical_coordinate(
        voxel_index_ijk=voxel_index_ijk,
        affine_ijk_to_xyz=affine_ijk_to_xyz,
    )
)  # [3]


# 계산 구조:
# X = -120 + 2 × 2.0 = -116.0 mm
# Y =  -90 + 3 × 1.5 =  -85.5 mm
# Z =  -60 + 4 × 3.0 =  -48.0 mm
print(
    "Voxel index [I, J, K]:",
    voxel_index_ijk,
)

print(
    "Physical coordinate [X, Y, Z] mm:",
    physical_coordinate_xyz_mm,
)

Voxel index [I, J, K]: tensor([2., 3., 4.], dtype=torch.float64)
Physical coordinate [X, Y, Z] mm: tensor([-116.0000,  -85.5000,  -48.0000], dtype=torch.float64)


In [16]:
# Cell 3 — Physical Coordinate를 Voxel Index로 역변환

def physical_coordinate_to_voxel_index(
    physical_coordinate_xyz_mm: torch.Tensor,  # [3] = [X, Y, Z] mm
    affine_ijk_to_xyz: torch.Tensor,           # [4, 4]
) -> torch.Tensor:                             # [3] = [I, J, K]
    """Physical coordinate를 affine 역행렬로 변환한 voxel index 반환."""
    
    # [X, Y, Z] → [X, Y, Z, 1]
    homogeneous_physical_coordinate = torch.cat(
        (
            physical_coordinate_xyz_mm,
            torch.ones(
                1,
                dtype=physical_coordinate_xyz_mm.dtype,
                device=physical_coordinate_xyz_mm.device,
            ),
        )
    ) # [4]
    
    # Voxel → Physical 변환의 반대 방향을 위한 affine inverse matrix
    inverse_affine_xyz_to_ijk = torch.linalg.inv(
        affine_ijk_to_xyz
    )  # [4, 4]
    
    # Physical coordinate를 연속적인 voxel index로 역변환
    homogeneous_voxel_index = (
        inverse_affine_xyz_to_ijk
        @ homogeneous_physical_coordinate
    )  # [4]
    
    # Homogeneous coordinate를 제외한 IJK index 추출
    continuous_voxel_index_ijk = (
        homogeneous_voxel_index[:3]
    )  # [3]

    return continuous_voxel_index_ijk

# Cell 2에서 계산한 physical coordinate를 voxel index로 복원
reconstructed_voxel_index_ijk = (
    physical_coordinate_to_voxel_index(
        physical_coordinate_xyz_mm=(
            physical_coordinate_xyz_mm
        ),  # [3]
        affine_ijk_to_xyz=(
            affine_ijk_to_xyz
        ),  # [4, 4]
    )
)  # [3]


# 원본 index와 복원 index의 axis별 오차 계산
round_trip_error_ijk = torch.abs(
    reconstructed_voxel_index_ijk
    - voxel_index_ijk
)  # [3]


# 가장 큰 axis 오차 추출
maximum_round_trip_error = torch.max(
    round_trip_error_ijk
).item()


print(
    "Original voxel index:",
    voxel_index_ijk,
)

print(
    "Physical coordinate:",
    physical_coordinate_xyz_mm,
)

print(
    "Reconstructed voxel index:",
    reconstructed_voxel_index_ijk,
)

print(
    "Round-trip error:",
    round_trip_error_ijk,
)

print(
    "Maximum round-trip error:",
    maximum_round_trip_error,
)

Original voxel index: tensor([2., 3., 4.], dtype=torch.float64)
Physical coordinate: tensor([-116.0000,  -85.5000,  -48.0000], dtype=torch.float64)
Reconstructed voxel index: tensor([2., 3., 4.], dtype=torch.float64)
Round-trip error: tensor([0., 0., 0.], dtype=torch.float64)
Maximum round-trip error: 0.0


In [17]:
# Cell 4 — Shape, Spacing과 Physical Extent 계산

def calculate_volume_geometry(
    array_shape_ijk: tuple[int, int, int],  # [I, J, K]
    affine_ijk_to_xyz: torch.Tensor,        # [4, 4]
) -> tuple[
    torch.Tensor,  # spacing [3]
    torch.Tensor,  # direction [3, 3]
    torch.Tensor,  # center span [3]
    torch.Tensor,  # field of view [3]
]:
    """Array shape와 affine으로 volume의 physical geometry 계산."""

    # Array shape를 physical 계산용 Tensor로 변환
    shape_ijk = torch.tensor(
        array_shape_ijk,
        dtype=affine_ijk_to_xyz.dtype,
        device=affine_ijk_to_xyz.device,
    )  # [3] = [I, J, K]
    
    # Affine에서 translation을 제외한 linear transform 추출
    linear_transform_ijk_to_xyz = (
        affine_ijk_to_xyz[:3, :3]
    )  # [3, 3]
    
    # 각 affine column의 Euclidean norm으로 voxel spacing 계산
    voxel_spacing_ijk_mm = torch.linalg.vector_norm(
        linear_transform_ijk_to_xyz,
        dim=0,
    ) # [3]
    
    # 각 affine column을 spacing으로 나눠 단위 direction vector 계산
    direction_ijk_to_xyz = (
        linear_transform_ijk_to_xyz
        / voxel_spacing_ijk_mm.unsqueeze(dim=0)
    )  # [3, 3]
    
    # 각 affine column의 Euclidean norm으로 voxel spacing 계산
    #
    # Rotation이 포함된 affine에서도 단순 diagonal 추출보다 안전
    voxel_spacing_ijk_mm = torch.linalg.vector_norm(
        linear_transform_ijk_to_xyz,
        dim=0,
    )  # [3]

    # 각 affine column을 spacing으로 나눠 단위 direction vector 계산
    direction_ijk_to_xyz = (
        linear_transform_ijk_to_xyz
        / voxel_spacing_ijk_mm.unsqueeze(dim=0)
    )  # [3, 3]

    # 첫 voxel center부터 마지막 voxel center까지의 거리
    center_to_center_span_ijk_mm = (
        shape_ijk - 1
    ) * voxel_spacing_ijk_mm  # [3]

    # 각 voxel이 차지하는 폭까지 포함한 전체 영상 범위: Voxel 개수 × voxel spacing
    field_of_view_ijk_mm = (
        shape_ijk
        * voxel_spacing_ijk_mm
    )  # [3]

    return (
        voxel_spacing_ijk_mm,
        direction_ijk_to_xyz,
        center_to_center_span_ijk_mm,
        field_of_view_ijk_mm,
    )


# NIfTI voxel array에서 axis별 shape 추출
array_shape_ijk: tuple[int, int, int] = (
    int(nifti_voxel_array.shape[0]),
    int(nifti_voxel_array.shape[1]),
    int(nifti_voxel_array.shape[2]),
)


# Shape와 affine을 이용한 volume geometry 계산
(
    voxel_spacing_ijk_mm,
    direction_ijk_to_xyz,
    center_to_center_span_ijk_mm,
    field_of_view_ijk_mm,
) = calculate_volume_geometry(
    array_shape_ijk=array_shape_ijk,
    affine_ijk_to_xyz=affine_ijk_to_xyz,
)


# 첫 voxel center의 index
first_voxel_index_ijk = torch.zeros(
    3,
    dtype=torch.float64,
)  # [3] = [0, 0, 0]


# 마지막 voxel center의 index
#
# Shape [6, 8, 10]
#       ↓ shape - 1
# Index [5, 7, 9]
last_voxel_index_ijk = torch.tensor(
    array_shape_ijk,
    dtype=torch.float64,
) - 1  # [3]


# 첫 번째와 마지막 voxel center의 physical coordinate 계산
first_voxel_center_xyz_mm = (
    voxel_index_to_physical_coordinate(
        voxel_index_ijk=first_voxel_index_ijk,
        affine_ijk_to_xyz=affine_ijk_to_xyz,
    )
)  # [3]

last_voxel_center_xyz_mm = (
    voxel_index_to_physical_coordinate(
        voxel_index_ijk=last_voxel_index_ijk,
        affine_ijk_to_xyz=affine_ijk_to_xyz,
    )
)  # [3]


print(
    "Array shape [I, J, K]:",
    array_shape_ijk,
)

print(
    "Voxel spacing [I, J, K] mm:",
    voxel_spacing_ijk_mm,
)

print("Direction matrix:")
print(direction_ijk_to_xyz)

print(
    "First voxel center [X, Y, Z] mm:",
    first_voxel_center_xyz_mm,
)

print(
    "Last voxel center [X, Y, Z] mm:",
    last_voxel_center_xyz_mm,
)

print(
    "Center-to-center span [I, J, K] mm:",
    center_to_center_span_ijk_mm,
)

print(
    "Field of view [I, J, K] mm:",
    field_of_view_ijk_mm,
)

Array shape [I, J, K]: (6, 8, 10)
Voxel spacing [I, J, K] mm: tensor([2.0000, 1.5000, 3.0000], dtype=torch.float64)
Direction matrix:
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]], dtype=torch.float64)
First voxel center [X, Y, Z] mm: tensor([-120.,  -90.,  -60.], dtype=torch.float64)
Last voxel center [X, Y, Z] mm: tensor([-110.0000,  -79.5000,  -33.0000], dtype=torch.float64)
Center-to-center span [I, J, K] mm: tensor([10.0000, 10.5000, 27.0000], dtype=torch.float64)
Field of view [I, J, K] mm: tensor([12., 12., 30.], dtype=torch.float64)
